In [21]:
# 標準
import os
from datetime import date, datetime

# サードパーティ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

%matplotlib inline

1.データ読み込み

In [2]:
# 使用するカラムを指定

usecols = ["Year", "Month", "Day", "Hour", "Minute", "Temperature", "Clearsky GHI", "Cloud Type", "Dew Point", "DHI", "DNI",
            "GHI", "Relative Humidity","Pressure", "Precipitable Water", "Wind Direction", "Wind Speed"]

In [3]:
# ファイルの読み込み＋複数年を合算


df_weather_yamanashi = pd.DataFrame()

for year in range(2016, 2021):

    file_path = f"../data/weather_yamanashi_{year}.csv"
    print(file_path)
    df = pd.read_csv(file_path, skiprows=2, usecols=usecols)

    if df_weather_yamanashi.empty:
        # print("empty")
        df_weather_yamanashi = df
    else:
        # print("not empty")
        df_weather_yamanashi =  pd.concat([df_weather_yamanashi, df])

df_weather_yamanashi.reset_index(drop = True)

../data/weather_yamanashi_2016.csv
../data/weather_yamanashi_2017.csv
../data/weather_yamanashi_2018.csv
../data/weather_yamanashi_2019.csv
../data/weather_yamanashi_2020.csv


,Year,Month,Day,Hour,Minute,Temperature,Clearsky GHI,Cloud Type,Dew Point,DHI,DNI,GHI,Relative Humidity,Pressure,Precipitable Water,Wind Direction,Wind Speed
0,2016,1,1,0,0,5.2,327.0,0.0,-5.2,65.0,801.0,327.0,47.19,1001.0,0.3,309.0,2.6
1,2016,1,1,1,0,6.7,463.0,0.0,-5.7,76.0,886.0,463.0,40.89,1001.0,0.3,310.0,2.7
2,2016,1,1,2,0,7.9,541.0,0.0,-5.5,80.0,916.0,541.0,38.04,1000.0,0.4,312.0,2.6
3,2016,1,1,3,0,8.7,563.0,0.0,-5.1,78.0,935.0,563.0,37.18,1000.0,0.4,311.0,2.4
4,2016,1,1,4,0,9.0,520.0,0.0,-4.7,74.0,919.0,520.0,37.59,1000.0,0.4,309.0,1.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43843,2020,12,31,19,0,-0.6,0.0,0.0,-6.4,0.0,0.0,0.0,64.98,986.0,0.5,267.0,1.7
43844,2020,12,31,20,0,-0.4,0.0,0.0,-6.1,0.0,0.0,0.0,65.48,987.0,0.5,272.0,1.7
43845,2020,12,31,21,0,-0.1,0.0,0.0,-5.9,0.0,0.0,0.0,65.05,987.0,0.5,278.0,1.9
43846,2020,12,31,22,0,1.0,0.0,0.0,-5.5,0.0,0.0,0.0,62.03,988.0,0.5,285.0,2.2


2.前処理

In [4]:
# 前処理用のdfを作成
df_weather_yamanashi_preprocessing = df_weather_yamanashi.copy()

In [5]:
# 時刻をつくる
# 元データがtz = 0なので、utc指定。
# df_weather_yamanashi_preprocessing["utc"] = pd.to_datetime(df_weather_yamanashi_preprocessing[["Year", "Month", "Day", "Hour"]], utc = True)

df_weather_yamanashi_preprocessing["utc"] = pd.to_datetime(df_weather_yamanashi_preprocessing[["Year", "Month", "Day", "Hour"]])


In [6]:
# time zoneの処理
# 元データはtz=0, local tz=9なので、hourを+9する。
# 23時とかは32時間とかになるからちゃんと計算する。

df_weather_yamanashi_preprocessing["jst"] = df_weather_yamanashi_preprocessing["utc"] + pd.Timedelta(hours=9)

In [ ]:
# day of yearを作成
# df_weather_yamanashi_preprocessing["day_of_year"] = df_weather_yamanashi_preprocessing["jst"].dt.dayofyear

In [8]:
# 時刻関係を作成
df_weather_yamanashi_preprocessing["month_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.month
df_weather_yamanashi_preprocessing["day_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.day
df_weather_yamanashi_preprocessing["hour_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.hour

In [13]:
#日射量から、発電量を作成
# 発電量 = GHI[W/㎡] ＊ 1 [hour] * 発電効率　＊　温度補正 * パネル面積
# 発電量[Wh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1
# 本来は外気気温ではなく、パネル温度で補正をかける

# 発電量[kWh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1/1000

# [Wh]
df_weather_yamanashi_preprocessing["pv"] = df_weather_yamanashi_preprocessing["GHI"] * 1 * 0.18 * (1 - 0.004 * (df_weather_yamanashi_preprocessing["Temperature"] - 25))  * 1


3.学習/検証/テストデータ分割

In [14]:
# 学習データ→2016年〜2018年
# 検証データ→2019年
# テストデータ→2020年

# GHIあり
feature_cols = ["pv", "GHI", "month_jst", "day_jst", "hour_jst", "Temperature"]

train_start = datetime(2016, 1, 1, 0, 0, 0)
# train_end = datetime(2018, 12, 31, 23, 59, 59)

val_start = datetime(2019, 1, 1, 0, 0, 0)
# val_end = datetime(2019, 12, 31, 23, 59, 59)

test_start = datetime(2020, 1, 1, 0, 0, 0)
# test_end = datetime(2020, 12, 31, 23, 59, 59)

df_train = df_weather_yamanashi_preprocessing[feature_cols][(df_weather_yamanashi_preprocessing["jst"] >= train_start ) 
                                           & (df_weather_yamanashi_preprocessing["jst"] < val_start)]
df_val = df_weather_yamanashi_preprocessing[feature_cols][(df_weather_yamanashi_preprocessing["jst"] >= val_start ) 
                                           & (df_weather_yamanashi_preprocessing["jst"] < test_start)]

df_test =  df_weather_yamanashi_preprocessing[feature_cols][df_weather_yamanashi_preprocessing["jst"] >= test_start]

In [ ]:
df_train

In [ ]:
df_val

4.ベースモデル構築

In [15]:
target = "pv"

# 線形回帰のインスタンス作成
lr = LinearRegression()


In [16]:
# 学習
X_train = df_train.drop(target, axis = 1)
Y_train =df_train[target]

lr.fit(X_train, Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [17]:
# 説明変数の係数
lr.coef_[0]

np.float64(0.18354887017226454)

In [18]:
# 切片
lr.intercept_

np.float64(1.6430843028405206)

In [19]:
# 検証
X_val = df_val.drop(target, axis = 1)
Y_val = df_val[target]

Y_pred_val = lr.predict(X_val)


In [22]:
# 検証の評価指標を確認

val_mae = mean_absolute_error(Y_val, Y_pred_val)
val_rmse = mean_squared_error(Y_val, Y_pred_val)

print(f"mae:{val_mae}. rmse:{val_rmse}\n")

mae:1.1619679654376016. rmse:2.4644182165614326



In [23]:
# テスト
X_test = df_test.drop(target, axis = 1)
Y_test = df_test[target]

Y_pred_test = lr.predict(X_test)

Y_pred_val = lr.predict(X_test)

5.KPI確認

In [27]:
# テストの評価指標を確認

test_mae = mean_absolute_error(Y_test, Y_pred_test)
test_rmse = mean_squared_error(Y_test, Y_pred_test)

print(f"mae:{test_mae}\nrmse:{test_rmse}")

mae:1.1849854150053822
rmse:2.5082657632374588
